Ce notebook permet de lancer un LDA uniquement sur les titres du sommaire et nn plus tout le contenu. 
Le but est d'obtenir des résultats mons bruités avec un sommaire qui porte déjà l'info nécessaire. 

In [1]:
!uv pip install -q nltk gensim pyLDAvis unidecode matplotlib seaborn pandas pyarrow

In [2]:
from gensim.models import CoherenceModel, LdaModel, LdaMulticore
from gensim.utils import simple_preprocess
from pathlib import Path
import gensim
import gensim.corpora as corpora
import json
import matplotlib.pyplot  as plt
import numpy as np
import os
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models as gensimvis
import warnings

In [3]:
INTERMEDIATE_DATA_DIR="intermediate_data"

# Utils LDA

In [4]:
def lda_model(processed_texts, num_topics=5, passes=10):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    warnings.filterwarnings('ignore')
    dictionary = corpora.Dictionary(processed_texts)
    corpus = [dictionary.doc2bow(text) for text in processed_texts]
    model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=passes ,workers=10, eta='auto' ,chunksize=1000)
    for topic in model.print_topics(num_words=5):
        print(topic)
    return model, corpus, dictionary
    
def visualize_lda(model, corpus, dictionary):
    pyLDAvis.enable_notebook()
    vis_data = gensimvis.prepare(model, corpus, dictionary)
    return pyLDAvis.display(vis_data)

In [5]:
def compute_coherence_values(dictionary, corpus, texts, max_topics=10):
    coherence_scores = []
    for num_topics in range(2, max_topics + 1):
        lda_model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=5 ,workers=10, eta='auto' ,chunksize=1000)
        coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_score = coherence_model.get_coherence()
        coherence_scores.append((num_topics, coherence_score))
        print(f"Num Topics: {num_topics}, Coherence Score: {coherence_score:.4f}")
    
    return coherence_scores

# Pour HS

In [26]:
df_hs = pd.read_parquet(f"{INTERMEDIATE_DATA_DIR}/processed_titles_hs.parquet")


In [29]:
all_chunks_hs = [list(title) for summary in df_hs["lda_documents"] for title in summary]


In [31]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs)

(0, '0.128*"travail" + 0.124*"temp" + 0.053*"jours" + 0.036*"organisation" + 0.019*"forfait"')
(1, '0.042*"application" + 0.040*"congés" + 0.034*"champ" + 0.031*"rémunération" + 0.030*"droit"')
(2, '0.088*"travail" + 0.084*"durée" + 0.079*"accord" + 0.030*"entreprise" + 0.022*"hebdomadaire"')
(3, '0.047*"repos" + 0.045*"période" + 0.040*"absence" + 0.035*"jours" + 0.035*"cours"')
(4, '0.152*"heures" + 0.095*"supplémentaires" + 0.041*"contingent" + 0.034*"salariés" + 0.026*"annuel"')


In [32]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

In [34]:
compute_coherence_values(dictionary_hs, corpus_hs, all_chunks_hs, max_topics=20)

Num Topics: 2, Coherence Score: 0.2808
Num Topics: 3, Coherence Score: 0.3306
Num Topics: 4, Coherence Score: 0.3653
Num Topics: 5, Coherence Score: 0.3330
Num Topics: 6, Coherence Score: 0.3493
Num Topics: 7, Coherence Score: 0.3610
Num Topics: 8, Coherence Score: 0.3662
Num Topics: 9, Coherence Score: 0.3811
Num Topics: 10, Coherence Score: 0.3781
Num Topics: 11, Coherence Score: 0.3706
Num Topics: 12, Coherence Score: 0.3802
Num Topics: 13, Coherence Score: 0.4104
Num Topics: 14, Coherence Score: 0.3886
Num Topics: 15, Coherence Score: 0.3785
Num Topics: 16, Coherence Score: 0.4031
Num Topics: 17, Coherence Score: 0.3837
Num Topics: 18, Coherence Score: 0.3919
Num Topics: 19, Coherence Score: 0.3979
Num Topics: 20, Coherence Score: 0.4013


[(2, 0.2807553619794412),
 (3, 0.330569096492268),
 (4, 0.36532756622578033),
 (5, 0.3330181489060591),
 (6, 0.34929581396230297),
 (7, 0.36096972326221444),
 (8, 0.3662344704810928),
 (9, 0.3810569930757082),
 (10, 0.3781479620493501),
 (11, 0.3706187241194599),
 (12, 0.38024203214928615),
 (13, 0.41042848305905955),
 (14, 0.3886112412172108),
 (15, 0.37853191653025825),
 (16, 0.4030667736139797),
 (17, 0.38370714636106895),
 (18, 0.3919232275987019),
 (19, 0.39794515724665047),
 (20, 0.4012778490051153)]

In [36]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs, num_topics= 13)

(0, '0.119*"jours" + 0.082*"travail" + 0.078*"temp" + 0.076*"forfait" + 0.033*"aménagement"')
(1, '0.085*"congés" + 0.050*"payés" + 0.049*"prime" + 0.033*"journée" + 0.032*"modalités"')
(2, '0.150*"durée" + 0.140*"travail" + 0.062*"hebdomadaire" + 0.046*"repos" + 0.024*"maximale"')
(3, '0.121*"repos" + 0.075*"jours" + 0.053*"prise" + 0.045*"publicité" + 0.041*"dépôt"')
(4, '0.142*"rémunération" + 0.051*"période" + 0.038*"lissage" + 0.030*"départs" + 0.029*"absence"')
(5, '0.124*"temp" + 0.119*"travail" + 0.081*"organisation" + 0.080*"application" + 0.064*"champ"')
(6, '0.118*"période" + 0.088*"référence" + 0.084*"absence" + 0.082*"cours" + 0.035*"contrat"')
(7, '0.273*"heures" + 0.172*"supplémentaires" + 0.081*"contingent" + 0.051*"annuel" + 0.039*"supplementaires"')
(8, '0.129*"disposition" + 0.045*"cadre" + 0.027*"applicables" + 0.027*"finale" + 0.025*"relative"')
(9, '0.154*"temp" + 0.066*"travail" + 0.060*"partiel" + 0.058*"salariés" + 0.026*"pause"')
(10, '0.203*"accord" + 0.058*"

In [37]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

Exception ignored in: <function ResourceTracker.__del__ at 0x7fad195cb6a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7f897a7df6a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7ffa32c236a0>
Traceback (most recent call last):
  File